# 5. Backtest an optimised portfolio against its benchmark

Needs the optimiser extra:

```
pip install "py-beacon[optimise]"
```

## The question this answers

An optimiser reports tracking error **ex ante** — what its risk model expects.
A backtest reports what **actually happened**. Those are different numbers, and
the gap between them is the honest measure of whether the risk model was any
good.

A constrained portfolio also trades differently: position and sector limits
force trades a cap-weighted index never makes, so turnover and costs rise. The
comparison here is deliberately like-for-like on costs, so any difference is
attributable to the constraints rather than to the fee schedule.

## Setup

In [ ]:
import logging

import numpy as np
import pandas as pd

from beacon.backtest.engine import BacktestEngine
from beacon.index.calculation import IndexCalculator
from beacon.index.constructor import IndexDefinition
from beacon.index.methodology import MarketCapWeighted
from beacon.optimise import (
    FullInvestment,
    GroupBounds,
    PositionBounds,
    minimise_tracking_error,
)
from beacon.risk import active_risk_contributions, estimate_risk_model
from beacon.synthetic import SyntheticConfig, generate

logging.basicConfig(level=logging.ERROR,
                    format="%(levelname)s %(name)s: %(message)s")

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

CAPITAL = 10_000_000.0
COSTS_BPS = 10.0
MAX_POSITION = 0.06
MAX_SECTOR = 0.25

## Step 1 — the benchmark, the risk model, the constraints

All three assembled in one cell — notebooks 01 and 04 cover them individually.

In [ ]:
CONFIG = SyntheticConfig(assets=40,
                         start="2021-01-04",
                         end="2024-12-31",
                         seed=3)

dataset = generate(CONFIG)
fetcher = dataset.fetcher()

definition = IndexDefinition(
    index_id="BENCH",
    index_name="BENCH Index",
    base_date=CONFIG.start,
    base_value=1000.0,
    currency=CONFIG.currency,
    eligibility_rules=[],
    weighting_scheme=MarketCapWeighted(use_free_float=True),
    rebalancing_frequency="QUARTERLY",
    universe_identifiers=list(dataset.universe.index),
    max_constituent_weight=0.10)

index = IndexCalculator(definition, fetcher).run(start_date=CONFIG.start,
                                                 end_date=CONFIG.end)

risk = estimate_risk_model(dataset.returns)

sectors = dataset.universe.groupby("SECTOR").groups
largest_sector = max(sectors, key=lambda name: len(sectors[name]))

constraints = [
    FullInvestment(),
    PositionBounds(0.0, MAX_POSITION),
    GroupBounds(largest_sector,
                [str(name) for name in sectors[largest_sector]],
                maximum=MAX_SECTOR),
]

print(f"{len(index.weight_snapshots)} rebalances to solve, "
      f"{risk.diagnostics.assets} assets, "
      f"{risk.diagnostics.observations} observations")

## Step 2 — optimise at *every* rebalance

This is the part that is easy to get wrong.

Optimising once and holding would compare a stale portfolio against a
rebalanced index, and then attribute to the constraints a difference that was
really staleness. Each rebalance gets its own solve against the same
constraints.

In [ ]:
schedule = {}
binding_counts: dict[str, int] = {}

for date, targets in sorted(index.weight_snapshots.items()):
    solution = minimise_tracking_error(targets, constraints, risk)
    schedule[date] = dict(solution.weights)

    for constraint in solution.binding:
        binding_counts[constraint.kind] = binding_counts.get(constraint.kind, 0) + 1

pd.Series({"rebalances": f"{len(schedule)}",
           "position limit": f"{MAX_POSITION:.0%}",
           "sector limit": f"{MAX_SECTOR:.0%} on {largest_sector}",
           "binding, by kind": str(binding_counts)},
          name="optimisation")

## Step 3 — backtest both, identically

Same engine, same capital, same cost schedule. The **only** difference is the
weight schedule: one tracks the index, the other tracks the optimised weights.

In [ ]:
plain = BacktestEngine(start_date=CONFIG.start, end_date=CONFIG.end,
                       initial_capital=CAPITAL, data_provider=fetcher,
                       index_result=index,
                       transaction_cost_bps=COSTS_BPS).run()

optimised = BacktestEngine(start_date=CONFIG.start, end_date=CONFIG.end,
                           initial_capital=CAPITAL, data_provider=fetcher,
                           target_weights=schedule,
                           transaction_cost_bps=COSTS_BPS).run()

AS_PERCENT = {"total_return", "annualised_return", "volatility",
              "max_drawdown", "tracking_error", "tracking_difference"}


def as_series(summary, name):
    """Format a summary, rendering absent metrics as "n/a" rather than NaN.

    The optimised run has no tracking metrics at all -- see the note below.
    """
    def show(key, value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return "n/a"

        return f"{value:.2%}" if key in AS_PERCENT else f"{value:.3f}"

    return pd.Series({key: show(key, value) for key, value in summary.items()},
                     name=name)


# Aligned on the union of both key sets and filled explicitly: the optimised
# summary omits the tracking metrics altogether, so leaving pd.concat to align
# them would manufacture a NaN that means "absent" while looking like "broken".
pd.concat([as_series(plain.summary(), "index"),
           as_series(optimised.summary(), "optimised")],
          axis=1).fillna("n/a")

Note the **`n/a` tracking metrics on the optimised run**. That is not a
failure: the index backtest was handed an `IndexResult`, so the engine knows
what it was trying to track and can measure the difference. The optimised run
was handed a raw weight schedule, which carries no benchmark, so there is
nothing to track *against*.

The two runs are compared directly in the next cell instead, and the tracking
error between them is computed by hand in step 4.

In [ ]:
plain_costs = sum(t.transaction_cost for t in plain.portfolio.transactions)
optimised_costs = sum(t.transaction_cost for t in optimised.portfolio.transactions)

plain_return = plain.trading_nav.iloc[-1] / CAPITAL - 1
optimised_return = optimised.trading_nav.iloc[-1] / CAPITAL - 1

pd.DataFrame({
    "index": {"final NAV": f"{plain.trading_nav.iloc[-1]:,.0f}",
              "trades": f"{len(plain.portfolio.transactions):,}",
              "costs paid": f"{plain_costs:,.0f}",
              "total return": f"{plain_return:.2%}"},
    "optimised": {"final NAV": f"{optimised.trading_nav.iloc[-1]:,.0f}",
                  "trades": f"{len(optimised.portfolio.transactions):,}",
                  "costs paid": f"{optimised_costs:,.0f}",
                  "total return": f"{optimised_return:.2%}"},
})

## Step 4 — ex ante versus realised

The number this notebook exists for.

In [ ]:
final_index = index.weight_snapshots[max(index.weight_snapshots)]
final_optimised = schedule[max(schedule)]

expected = active_risk_contributions(final_optimised, final_index, risk.covariance)

difference = (optimised.trading_nav.pct_change()
              - plain.trading_nav.pct_change()).dropna()
realised = float(difference.std() * np.sqrt(252))

pd.Series({"ex ante (risk model)": f"{expected.volatility:.2%}",
           "realised (backtest)": f"{realised:.2%}",
           "ratio": f"{realised / expected.volatility:.2f}x"},
          name="tracking error")

These are computed from different things over different windows: the first is
what the covariance predicted at the **final** rebalance, the second is what the
two NAV paths actually did over the **whole run**. They will not agree, and they
are not supposed to.

The size of the gap is the useful signal, not its existence. A realised figure
several times the forecast means the risk model was not describing this
portfolio's actual behaviour.

### How it moved over time

A single realised number hides everything interesting. Rolling it over a
quarter shows whether the tracking error was stable or concentrated in a few
episodes:

In [ ]:
rolling = difference.rolling(63).std() * np.sqrt(252)

rolling.dropna().iloc[::42].to_frame("realised tracking error").style.format("{:.2%}")

Tracking error is rarely constant. It widens when correlations move — which is
exactly when a risk model estimated over a calmer period is least reliable, and
exactly when you most want it to hold.

## Step 5 — what the constraints cost

In [ ]:
extra_trades = (len(optimised.portfolio.transactions)
                - len(plain.portfolio.transactions))

pd.Series({"return difference": f"{optimised_return - plain_return:+.2%}",
           "extra trading paid": f"{optimised_costs - plain_costs:+,.0f}",
           "extra trades": f"{extra_trades:+,}"},
          name="cost of the constraints")

**Read the return difference carefully.**

Constraints cost turnover — though look closely at *how*. On this run the trade
**count** is identical; what changed is their **size**. Holding a 6% cap
against an index whose largest name runs at 10% means trading away from the
benchmark at every rebalance and back again as prices move, so the same
number of trades moves more notional and the costs line rises. That number is
a real, repeatable cost of the policy.

What they do to **return** is not reliable in either direction. On this seed the
constrained portfolio came out one way; another seed can go the other. That is
one path, not a finding.

Constraints are a risk decision. Judging them by a single realised return is
precisely how a backtest misleads — the cost is knowable in advance, the
benefit is not.

## Where to go next

You have now been through every layer:

| Notebook | Covers |
| --- | --- |
| `01_index_and_backtest` | methodology → calculator → backtest |
| `02_backtest_analysis` | statistics, drift, attribution, charts |
| `03_index_futures` | cost of carry, basis, DV01, rolling |
| `04_optimised_index` | constraints, and which of them bind |
| `05_optimised_backtest` | ex ante versus realised |